In [12]:
import pandas as pd
import numpy as np

print("=" * 60)
print("🚗 USED CAR PRICE OPTIMIZER - DATA PREPARATION")
print("=" * 60)

# ============================================
# STEP 1: Kaggle Data Load Karo
# ============================================
print("\n📥 STEP 1: Loading Kaggle dataset...")
df = pd.read_csv('used_cars.csv')

print(f"   ✅ Loaded: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"   Columns: {df.columns.tolist()}")

# ============================================
# STEP 2: Price Clean Karo
# ============================================
print("\n🧹 STEP 2: Cleaning price column...")
print(f"   Before: {df['price'].head(2).tolist()}")

df['price'] = df['price'].str.replace('$', '', regex=False)
df['price'] = df['price'].str.replace(',', '', regex=False)
df['price'] = pd.to_numeric(df['price'], errors='coerce')

print(f"   After: {df['price'].head(2).tolist()}")

# ============================================
# STEP 3: Column Names Fix Karo
# ============================================
print("\n📝 STEP 3: Fixing column names...")
df.columns = df.columns.str.lower().str.replace(' ', '_')

if 'milage' in df.columns:
    df.rename(columns={'milage': 'mileage'}, inplace=True)

print(f"   New columns: {df.columns.tolist()}")

# ============================================
# STEP 4: Mileage Clean Karo ("51,000 mi." → 51000)
# ============================================
print("\n🔧 STEP 4: Cleaning mileage...")
print(f"   Before: {df['mileage'].head(2).tolist()}")

df['mileage'] = df['mileage'].astype(str)
df['mileage'] = df['mileage'].str.replace(',', '', regex=False)
df['mileage'] = df['mileage'].str.extract(r'(\d+)').astype(float)

print(f"   After: {df['mileage'].head(2).tolist()}")

# ============================================
# STEP 5: Engine Se Info Extract Karo
# ============================================
print("\n🔧 STEP 5: Extracting engine info...")
print(f"   Example: {df['engine'].iloc[0][:80]}...")

# Horsepower
df['horsepower'] = df['engine'].str.extract(r'(\d+\.?\d*)HP').astype(float)

# Engine size (Liters)
df['engine_size'] = df['engine'].str.extract(r'(\d+\.?\d*)\s*L').astype(float)

# Cylinders
# Cylinders - V6, I4, V8 se number nikalo
df['cylinders'] = df['engine'].str.extract(r'V(\d+)').astype(float)

print(f"   Extracted: HP, engine_size(L), cylinders")
print(f"   Sample: HP={df['horsepower'].iloc[0]}, Size={df['engine_size'].iloc[0]}L, Cyl={df['cylinders'].iloc[0]}")

# Original engine column hatao
df.drop('engine', axis=1, inplace=True)

# ============================================
# STEP 6: Text Columns Ko Numbers Mein Badlo
# ============================================
print("\n🔄 STEP 6: Converting text columns to numbers...")

# Brand → count encoding (jitni zyada listings, utna bada number)
brand_counts = df['brand'].value_counts()
df['brand_code'] = df['brand'].map(brand_counts)
print(f"   Brand: {df['brand'].iloc[0]} → {df['brand_code'].iloc[0]}")
df.drop('brand', axis=1, inplace=True)

# Model → count encoding
model_counts = df['model'].value_counts()
df['model_code'] = df['model'].map(model_counts)
print(f"   Model: {df['model'].iloc[0][:30]}... → {df['model_code'].iloc[0]}")
df.drop('model', axis=1, inplace=True)

# Fuel type → simple number
fuel_types = {name: idx for idx, name in enumerate(df['fuel_type'].unique())}
df['fuel_code'] = df['fuel_type'].map(fuel_types)
print(f"   Fuel: {df['fuel_type'].iloc[0]} → {df['fuel_code'].iloc[0]}")
df.drop('fuel_type', axis=1, inplace=True)

# Transmission → simple number
trans_types = {name: idx for idx, name in enumerate(df['transmission'].unique())}
df['transmission_code'] = df['transmission'].map(trans_types)
print(f"   Transmission: {df['transmission'].iloc[0]} → {df['transmission_code'].iloc[0]}")
df.drop('transmission', axis=1, inplace=True)

# Exterior color → count
ext_counts = df['ext_col'].value_counts()
df['ext_col_code'] = df['ext_col'].map(ext_counts)
df.drop('ext_col', axis=1, inplace=True)

# Interior color → count
int_counts = df['int_col'].value_counts()
df['int_col_code'] = df['int_col'].map(int_counts)
df.drop('int_col', axis=1, inplace=True)

# Accident → 1 (yes) or 0 (no)
df['accident_flag'] = df['accident'].str.contains('accident|damage', case=False, na=False).astype(int)
print(f"   Accident: {df['accident'].iloc[0][:30]}... → {df['accident_flag'].iloc[0]}")
df.drop('accident', axis=1, inplace=True)

# Clean title → 1 (yes) or 0 (no)
df['clean_title_flag'] = df['clean_title'].map({'Yes': 1}).fillna(0).astype(int)
print(f"   Clean title: {df['clean_title'].iloc[0]} → {df['clean_title_flag'].iloc[0]}")
df.drop('clean_title', axis=1, inplace=True)

# ============================================
# STEP 7: Missing Values Bharo
# ============================================
print("\n🔧 STEP 7: Filling missing values...")
before = df.isna().sum().sum()
numeric_cols = df.select_dtypes(include=[np.number]).columns
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())
after = df.isna().sum().sum()
print(f"   Missing before: {before} → after: {after}")

# ============================================
# STEP 8: Price Outliers Hatao
# ============================================
print("\n✂️ STEP 8: Removing extreme prices...")
before = len(df)
lower = df['price'].quantile(0.01)
upper = df['price'].quantile(0.99)
df = df[(df['price'] > lower) & (df['price'] < upper)]
print(f"   Removed {before - len(df)} outliers. Remaining: {len(df)} rows")

# ============================================
# ============================================
# STEP 9: BUSINESS LOGIC COLUMNS ADD KARO
# ============================================
print("\n💰 STEP 9: Adding business columns...")

# Cost price: Dealer ne 20% kam mein khareedi
df['cost_price'] = (df['price'] * 0.80).round(0)
print(f"   cost_price added (price × 0.80)")

# 3 fake competitors
df['comp1_price'] = (df['price'] * np.random.uniform(0.90, 1.10, len(df))).round(0)
df['comp2_price'] = (df['price'] * np.random.uniform(0.90, 1.10, len(df))).round(0)
df['comp3_price'] = (df['price'] * np.random.uniform(0.90, 1.10, len(df))).round(0)
df['competitor_avg'] = df[['comp1_price', 'comp2_price', 'comp3_price']].mean(axis=1)
print(f"   3 competitors + competitor_avg added")

# ============================================
# UNITS SOLD — SIMPLE CLEAR FORMULA
# ============================================

# Step A: Price difference from competitor (negative = tum saste, positive = tum mehnga)
df['price_diff'] = df['competitor_avg'] - df['price']

# Step B: Base demand 50 units
# Step C: Har $1000 saste pe +5 extra demand
# Step D: Har $1000 mehnga pe -5 demand
# Step E: Better condition = +demand, higher mileage = -demand

df['units_sold'] = 50  # base

# Price difference effect
df['units_sold'] += (df['price_diff'] / 1000) * 5

# Model year effect (newer = more demand)
df['units_sold'] += (df['model_year'] - 2015) * 2

# Mileage effect (more miles = less demand)
df['units_sold'] -= (df['mileage'] / 10000) * 1

# Horsepower effect (more HP = sporty = niche buyers, slightly less demand)
df['units_sold'] -= (df['horsepower'] / 100) * 0.5

# Small random noise (5% variation)
df['units_sold'] *= np.random.normal(1, 0.05, len(df))

# Round and clip
df['units_sold'] = df['units_sold'].clip(lower=5, upper=200).round(0).astype(int)

# Price diff column hatao (feature mein nahi chahiye alag se)
df.drop('price_diff', axis=1, inplace=True)

print(f"   units_sold added (deterministic formula)")
print(f"   Sample: Price=${df['price'].iloc[0]:,.0f}, CompAvg=${df['competitor_avg'].iloc[0]:,.0f}, Sold={df['units_sold'].iloc[0]} units")
print(f"   y range: {df['units_sold'].min()} - {df['units_sold'].max()}")
print(f"   y mean: {df['units_sold'].mean():.0f}")
# ============================================
# STEP 10: Final Save
# ============================================
print("\n💾 STEP 10: Saving prepared data...")
df.to_csv('used_cars_prepared.csv', index=False)

# Feature list save karo
exclude = ['price', 'cost_price', 'comp1_price', 'comp2_price', 'comp3_price', 'units_sold']
feature_cols = [col for col in df.select_dtypes(include=[np.number]).columns if col not in exclude]

import json
with open('feature_cols.json', 'w') as f:
    json.dump(feature_cols, f)

# ============================================
# FINAL SUMMARY
# ============================================
print("\n" + "=" * 60)
print("✅ DATA PREPARATION COMPLETE!")


🚗 USED CAR PRICE OPTIMIZER - DATA PREPARATION

📥 STEP 1: Loading Kaggle dataset...
   ✅ Loaded: 4009 rows, 12 columns
   Columns: ['brand', 'model', 'model_year', 'milage', 'fuel_type', 'engine', 'transmission', 'ext_col', 'int_col', 'accident', 'clean_title', 'price']

🧹 STEP 2: Cleaning price column...
   Before: ['$10,300', '$38,005']
   After: [10300, 38005]

📝 STEP 3: Fixing column names...
   New columns: ['brand', 'model', 'model_year', 'mileage', 'fuel_type', 'engine', 'transmission', 'ext_col', 'int_col', 'accident', 'clean_title', 'price']

🔧 STEP 4: Cleaning mileage...
   Before: ['51,000 mi.', '34,742 mi.']
   After: [51000.0, 34742.0]

🔧 STEP 5: Extracting engine info...
   Example: 300.0HP 3.7L V6 Cylinder Engine Flex Fuel Capability...
   Extracted: HP, engine_size(L), cylinders
   Sample: HP=300.0, Size=3.7L, Cyl=6.0

🔄 STEP 6: Converting text columns to numbers...
   Brand: Ford → 386
   Model: Utility Police Interceptor Bas... → 3
   Fuel: E85 Flex Fuel → 0
   Transmi

In [16]:
import pandas as pd
import numpy as np
import json
import pickle
import warnings
warnings.filterwarnings('ignore')

print("=" * 60)
print("🤖 MODEL TRAINING START")
print("=" * 60)

# ============================================
# STEP 1: Prepared Data Load Karo
# ============================================
print("\n📥 Loading prepared data...")
df = pd.read_csv('used_cars_prepared.csv')
print(f"   Rows: {len(df):,}")

# Feature list load karo
with open('feature_cols.json', 'r') as f:
    FEATURES = json.load(f)

print(f"   Features ({len(FEATURES)}): {FEATURES}")

# ============================================
# STEP 2: X aur Y Define Karo
# ============================================
print("\n🎯 Defining X (features) and y (target)...")

X = df[FEATURES].copy()
y = df['units_sold'].copy()

print(f"   X shape: {X.shape}")
print(f"   y shape: {y.shape}")
print(f"   y range: {y.min()} - {y.max()} units")
print(f"   y mean: {y.mean():.0f} units")

# ============================================
# STEP 3: Train/Test Split
# ============================================
print("\n✂️ Splitting data (80% train, 20% test)...")
split_idx = int(len(X) * 0.80)

X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f"   Train: {len(X_train):,} rows")
print(f"   Test: {len(X_test):,} rows")

# ============================================
# STEP 4: Model Train Karo
# ============================================
print("\n🧠 Training XGBoost model...")
from xgboost import XGBRegressor

model = XGBRegressor(
    n_estimators=200,       # 200 trees
    max_depth=5,            # Tree depth
    learning_rate=0.05,     # Slow learning
    random_state=42,
    verbosity=0
)

model.fit(X_train, y_train)
print("   ✅ Training complete!")

# ============================================
# STEP 5: Evaluate Karo
# ============================================
print("\n📊 Evaluating model...")
from sklearn.metrics import mean_absolute_error, r2_score

# Predictions
train_pred = model.predict(X_train)
test_pred = model.predict(X_test)

# Metrics
train_mae = mean_absolute_error(y_train, train_pred)
test_mae = mean_absolute_error(y_test, test_pred)
train_r2 = r2_score(y_train, train_pred)
test_r2 = r2_score(y_test, test_pred)

print(f"\n   Training Performance:")
print(f"   ├── MAE: {train_mae:.1f} units")
print(f"   └── R² Score: {train_r2:.3f}")

print(f"\n   Test Performance:")
print(f"   ├── MAE: {test_mae:.1f} units")
print(f"   └── R² Score: {test_r2:.3f}")

# Accuracy interpretation
accuracy = 100 - (test_mae / y_test.mean() * 100)
print(f"\n   📈 Prediction Accuracy: ~{accuracy:.0f}%")
print(f"   (Average error is {test_mae:.0f} units out of {y_test.mean():.0f} avg demand)")

# ============================================
# STEP 6: Feature Importance
# ============================================
print("\n⭐ Top 10 Important Features:")
importance = pd.DataFrame({
    'feature': FEATURES,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

for i, row in importance.head(10).iterrows():
    bar = '█' * int(row['importance'] * 100)
    print(f"   {row['feature']:20s}: {row['importance']:.3f} {bar}")

# ============================================
# STEP 7: Model + Info Save Karo
# ============================================
print("\n💾 Saving model...")
with open('demand_model.pkl', 'wb') as f:
    pickle.dump({
        'model': model,
        'features': FEATURES,
        'metrics': {
            'mae': test_mae,
            'r2': test_r2,
            'accuracy': accuracy
        }
    }, f)

# Feature ranges save karo (slider limits ke liye)
feature_ranges = {}
for col in FEATURES:
    feature_ranges[col] = {
        'min': float(df[col].min()),
        'max': float(df[col].max()),
        'avg': float(df[col].mean())
    }

with open('feature_ranges.json', 'w') as f:
    json.dump(feature_ranges, f, indent=2)

print("   ✅ Model saved: models/demand_model.pkl")
print("   ✅ Feature ranges saved: models/feature_ranges.json")

# ============================================
# SUMMARY
# ============================================
print("\n" + "=" * 60)
print("✅ MODEL TRAINING COMPLETE!")
print("=" * 60)
print(f"📊 Test MAE: {test_mae:.1f} units")
print(f"📊 Test R²: {test_r2:.3f}")
print(f"📁 Files: demand_model.pkl")


🤖 MODEL TRAINING START

📥 Loading prepared data...
   Rows: 3,923
   Features (14): ['model_year', 'mileage', 'horsepower', 'engine_size', 'cylinders', 'brand_code', 'model_code', 'fuel_code', 'transmission_code', 'ext_col_code', 'int_col_code', 'accident_flag', 'clean_title_flag', 'competitor_avg']

🎯 Defining X (features) and y (target)...
   X shape: (3923, 14)
   y shape: (3923,)
   y range: 5 - 131 units
   y mean: 43 units

✂️ Splitting data (80% train, 20% test)...
   Train: 3,138 rows
   Test: 785 rows

🧠 Training XGBoost model...
   ✅ Training complete!

📊 Evaluating model...

   Training Performance:
   ├── MAE: 4.2 units
   └── R² Score: 0.880

   Test Performance:
   ├── MAE: 5.9 units
   └── R² Score: 0.734

   📈 Prediction Accuracy: ~86%
   (Average error is 6 units out of 43 avg demand)

⭐ Top 10 Important Features:
   model_year          : 0.634 ███████████████████████████████████████████████████████████████
   cylinders           : 0.064 ██████
   mileage             :